# 逐日研究续接的文件复核
人工账本只验证工程约束；实际资料只记录阻断，没有真实策略收益。代码单元由项目Python进程依序执行，输出为实际捕获；没有使用Jupyter内核。

In [1]:
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
root = Path('/Users/guo/Documents/stocks/quant-research')
audit = root / 'artifacts/earnings-validation-20261012/daily-audit-v1'
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
report = json.loads((audit / 'audit.json').read_text())
for name, expected in report['source_hashes'].items():
    assert sha(name) == expected, name
print(json.dumps({'bound_sources_verified': len(report['source_hashes']), 'formal_profit_qualified': report['formal_profit_qualified']}))

{"bound_sources_verified": 6, "formal_profit_qualified": false}


In [2]:
journal = audit / 'engineering-journal'
rows = [json.loads(line) for line in (journal / 'journal.jsonl').read_text().splitlines()]
accepted = [row for row in rows if row['status'] == 'engineering_replay_committed']
assert len(accepted) == 2 and rows[-1]['status'] == 'rejected_attempt'
old, new = [journal / 'attempts' / row['attempt_id'] for row in accepted]
end = json.loads((old / 'input/inputs.json').read_text())['end']
checked = 0
for path in sorted((old / 'result').glob('h*/**/summary.json')):
    folder = path.parent.relative_to(old / 'result')
    before = pd.read_parquet(path.with_name('nav.parquet'))
    after = pd.read_parquet(new / 'result' / folder / 'nav.parquet')
    prefix = after.loc[after.date.le(end)].reset_index(drop=True)
    assert before.date.tolist() == prefix.date.tolist()
    for column in ['nav', 'cash', 'market_value', 'receivables', 'dividend_tax_reserve']:
        assert np.array_equal(before[column].to_numpy(), prefix[column].to_numpy()), (folder, column)
    assert len(after) == len(before) + 1
    checked += 1
assert checked == 36
pairs_old = pd.read_parquet(old / 'result/h20/frozen-pairs.parquet')
pairs_new = pd.read_parquet(new / 'result/h20/frozen-pairs.parquet')
pd.testing.assert_frame_equal(pairs_old, pairs_new, check_exact=True)
pending = pd.read_parquet(old / 'result/h20/base/E1/pending_entries.parquet')
fills = pd.read_parquet(new / 'result/h20/base/E1/fills.parquet')
assert set(pending.trade_id) <= set(fills.trade_id)
assert accepted[-1]['record_sha256'] == report['last_accepted_engineering_head']
print(json.dumps({'artificial_account_prefixes_verified': checked, 'old_sessions': len(before), 'new_sessions': len(after), 'frozen_pairs': len(pairs_old), 'old_E1_pending_orders_filled': len(pending), 'historical_revision_status': rows[-1]['status']}))

{"artificial_account_prefixes_verified": 36, "old_sessions": 31, "new_sessions": 32, "frozen_pairs": 3, "old_E1_pending_orders_filled": 1, "historical_revision_status": "rejected_attempt"}


In [3]:
observation = report['actual_observation']
assert observation['account_state'] == 'not_run'
assert observation['candidate_state'] == 'not_evaluated_source_rejected'
assert not observation['cash_day_claim'] and not observation['prospective_start_registered']
assert pd.Timestamp(observation['recorded_at']) < pd.Timestamp(observation['primary_deadline'])
assert pd.Timestamp(observation['source_created_at']) <= pd.Timestamp(observation['clock_checked_at'])
print(json.dumps({key: observation[key] for key in ['signal_date', 'recorded_at', 'primary_deadline', 'status', 'blockers', 'account_state', 'cash_day_claim', 'prospective_start_registered']}, ensure_ascii=False))

{"signal_date": "2026-10-12", "recorded_at": "2026-10-12T16:09:15.178573+00:00", "primary_deadline": "2026-10-13T09:00:00+08:00", "status": "real_input_observation_blocked", "blockers": ["independent_source_acceptance_and_registration_required", "training_snapshot_cannot_enter_account", "minimum_commission_unconfirmed"], "account_state": "not_run", "cash_day_claim": false, "prospective_start_registered": false}
